# **Άσκηση 2.**

Import των Data.

In [2]:
import pandas as pd
import zipfile
import os

zip_path = '/content/Data_Receptors.zip'

# Αποσυμπίεση του zip αρχείου και φόρτωση των δεδομένων
with zipfile.ZipFile(zip_path, 'r') as zip_ref:

    zip_ref.extractall('/content')

    # Φόρτωση των δεδομένων εκπαίδευσης και τεστ
    train_features = pd.read_csv('/content/Train_Features.csv')
    train_labels = pd.read_csv('/content/Train_Labels.csv')
    test_features = pd.read_csv('/content/Test_Features.csv')

# Εμφάνιση των πρώτων γραμμών από κάθε DataFrame για επιβεβαίωση
print("Train Features:")
print(train_features.head())
print("\nTrain Labels:")
print(train_labels.head())
print("\nTest Features:")
print(test_features.head())


Train Features:
   28.406  22.081  0  0.1  45.633  2.4611  4.9222  45.633.1  1.2333  4.5185  \
0  30.579  22.122  0    3  51.506  2.4802  4.9235    51.506  1.3207  4.5949   
1  29.165  21.984  0    3  48.652  2.4805  4.9240    48.652  1.3149  4.5486   
2  23.201  17.981  0    1  39.309  2.4240  4.8126    39.309  1.3555  4.3224   
3  27.203  21.749  0    1  45.762  2.4490  4.8979    45.762  1.2712  4.4889   
4  25.272  19.676  0    1  41.444  2.4181  4.8361    41.444  1.2559  4.4058   

   ...  0.1354  0.1355  0.1356  1.1003  0.1357  0.1358  0.1359  0.1360  \
0  ...       0       1       0       1       0       0       0       0   
1  ...       0       1       0       1       0       0       0       0   
2  ...       1       0       0       1       0       0       0       0   
3  ...       1       1       0       1       0       0       0       0   
4  ...       1       0       0       1       0       0       0       0   

   1.1004  1.1005  
0       1       1  
1       1       1  
2   

**Ερώτημα Α και Β.**

In [3]:
!pip install keras-tuner

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.9/128.9 kB 2.9 MB/s eta 0:00:00


In [4]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
from keras.models import Sequential
from keras.layers import Dense, Conv1D, Flatten, MaxPooling1D, Dropout
import kerastuner as kt


# Διαχωρισμός σε συνεχή και δυαδικά χαρακτηριστικά
continuous_features = train_features.iloc[:, :1425]
binary_features = train_features.iloc[:, 1425:]

# Προεπεξεργασία δεδομένων
scaler = StandardScaler()
continuous_features_scaled = scaler.fit_transform(continuous_features)

# Επανασύνθεση των κλιμακωμένων συνεχών και δυαδικών χαρακτηριστικών
train_features_scaled = np.concatenate([continuous_features_scaled, binary_features], axis=1)

# PCA για μείωση διαστάσεων
pca = PCA(n_components=0.95)
train_features_pca = pca.fit_transform(train_features_scaled)

# Μετατροπή δεδομένων για είσοδο στο CNN
train_features_pca = np.expand_dims(train_features_pca, axis=2)

# Διαχωρισμός δεδομένων εκπαίδευσης και ελέγχου
X_train, X_test, y_train, y_test = train_test_split(train_features_pca, train_labels, test_size=0.2, random_state=42)

# Καθορισμός της αρχιτεκτονικής του μοντέλου μέσω Keras Tuner
def build_model(hp):
    model = Sequential()
    model.add(Conv1D(filters=hp.Int('input_units', min_value=32, max_value=128, step=32),
                     kernel_size=3, activation='relu', input_shape=(X_train.shape[1], 1)))
    model.add(MaxPooling1D(pool_size=2))
    for i in range(hp.Int('n_layers', 1, 3)):
        model.add(Conv1D(filters=hp.Int(f'conv_{i}_units', min_value=32, max_value=128, step=32),
                         kernel_size=3, activation='relu'))
        model.add(MaxPooling1D(pool_size=2))
    model.add(Flatten())
    model.add(Dense(hp.Int('dense_units', min_value=32, max_value=128, step=32), activation='relu'))
    model.add(Dropout(hp.Float('dropout', min_value=0, max_value=0.5, step=0.1)))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# εκτέλεση της αναζήτησης
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=15,
    executions_per_trial=3,
    directory='my_dir',
    project_name='hparam_tuning'
)

tuner.search_space_summary()

tuner.search(X_train, y_train, epochs=15, validation_data=(X_test, y_test))

# Εμφάνιση των καλύτερων μοντέλων
tuner.results_summary()

# Λήψη του καλύτερου μοντέλου και εκπαίδευσή του
best_model = tuner.get_best_models(num_models=1)[0]
best_model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))

# Αξιολόγηση του μοντέλου
loss, accuracy = best_model.evaluate(X_test, y_test)
print(f'Accuracy: {accuracy*100}%')


Trial 15 Complete [00h 00m 49s]
val_accuracy: 0.9192824959754944

Best val_accuracy So Far: 0.9252615769704183
Total elapsed time: 00h 08m 50s
Results summary
Results in my_dir/hparam_tuning
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 13 summary
Hyperparameters:
input_units: 32
n_layers: 3
conv_0_units: 128
dense_units: 96
dropout: 0.30000000000000004
conv_1_units: 64
conv_2_units: 96
Score: 0.9252615769704183

Trial 02 summary
Hyperparameters:
input_units: 64
n_layers: 3
conv_0_units: 128
dense_units: 96
dropout: 0.1
conv_1_units: 96
conv_2_units: 128
Score: 0.9237668116887411

Trial 07 summary
Hyperparameters:
input_units: 32
n_layers: 1
conv_0_units: 96
dense_units: 96
dropout: 0.30000000000000004
conv_1_units: 32
conv_2_units: 64
Score: 0.9237667918205261

Trial 08 summary
Hyperparameters:
input_units: 32
n_layers: 2
conv_0_units: 64
dense_units: 128
dropout: 0.30000000000000004
conv_1_units: 128
conv_2_units: 64
Score: 0.9222720464070638

Trial 09

Βέλτιστες υπερπαραμέτροι

In [6]:
import kerastuner as kt
import tensorflow as tf

# Εκτύπωση των καλύτερων υπερπαραμέτρων
best_hyperparameters = tuner.get_best_hyperparameters(num_trials=1)[0]

print("Οι βέλτιστες υπερπαράμετροι:")
for param in best_hyperparameters.space:
    print(f"{param.name}: {best_hyperparameters.get(param.name)}")


Οι βέλτιστες υπερπαράμετροι:
input_units: 32
n_layers: 3
conv_0_units: 128
dense_units: 96
dropout: 0.30000000000000004
conv_1_units: 64
conv_2_units: 96


**Ερώτημα Γ.**

In [8]:
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
import numpy as np

# Αξιολόγηση του μοντέλου στο σετ ελέγχου
loss, accuracy = best_model.evaluate(X_test, y_test)
print(f'Accuracy: {accuracy*100}%')

# Υπολογισμός του AUC
y_pred = best_model.predict(X_test).ravel()
auc = roc_auc_score(y_test, y_pred)
print(f'AUC: {auc}')

# Προετοιμασία των δεδομένων του test set (παρόμοια με την προετοιμασία του train set)
continuous_features_test = test_features.iloc[:, :1425]
binary_features_test = test_features.iloc[:, 1425:]
scaler = StandardScaler()
continuous_features_test_scaled = scaler.fit_transform(continuous_features_test)
test_features_scaled = np.concatenate([continuous_features_test_scaled, binary_features_test], axis=1)
test_features_pca = pca.transform(test_features_scaled)
test_features_pca = np.expand_dims(test_features_pca, axis=2)

# Πρόβλεψη του μοντέλου για το test set
test_predictions = best_model.predict(test_features_pca).ravel()

# Δημιουργία DataFrame με τις προβλέψεις
test_predictions_df = pd.DataFrame({'predicted_label': (test_predictions > 0.5).astype(int),
                                    'prediction_score': test_predictions})

# Αποθήκευση των προβλέψεων σε αρχείο CSV
test_predictions_df.to_csv('test_predictions.csv', index=False)


7/7 [==============================] - 0s 5ms/step - loss: 0.3258 - accuracy: 0.9058
Accuracy: 90.58296084403992%
7/7 [==============================] - 0s 6ms/step
AUC: 0.9509803921568627
4/4 [==============================] - 0s 7ms/step
